In [1]:
import time
import jax
import jax.numpy as jnp
import pennylane as qml

# ==========================================
# HARDWARE HARDENING CONFIGURATION
# ==========================================
# LAPTOP: Set to False (cuts RAM usage in half)
# HPC LEONARDO: Set to True (harnesses A100 double-precision hardware)
jax.config.update("jax_enable_x64", True)


# Optional: Verify JAX sees the A100 GPU before running
print(f"JAX running on: {jax.devices()}")

def prepare_superposition_vectors(x, log_length_sequences, log_feature_dimension, non_linear_order):
    """
    Pre-computes the normalized superposition states psi_j for step (2) 
    using 1 + non_linear_order data registers (A + copies of B for the non-linear coefficient)
    For each control index j, creates: sum_{i <= j} |x_i>^{otimes (non_linear_order + 1)} normalized
    Pre-computing these vectors ensures the JAX JIT compiler can optimize the QNode without overhead.
    """
    length_sequences = 2**log_length_sequences
    states_j = []
    
    for j in range(length_sequences):
        # Target dimension: (2**log_feature_dimension)**(non_linear_order + 1)
        dim = 2**(log_feature_dimension * (non_linear_order + 1))
        vec_j = jnp.zeros(dim, dtype=jnp.complex64)
        
        for i in range(j + 1):
            xi = x[i]
            # Compute the tensor product power: x_i^{\otimes (non_linear_order + 1)}
            xi_power = xi
            for _ in range(non_linear_order):
                xi_power = jnp.kron(xi_power, xi)
            vec_j += xi_power
            
        # Normalize the superposition state vector
        norm = jnp.linalg.norm(vec_j)
        vec_j = vec_j / (norm + 1e-12)
        states_j.append(vec_j)
        
    return states_j

def circuit_G_block(q1, q2, block_weights):
    """
    Applies your basic 2-qubit block using a pre-sliced row of 12 weights.
    No 'idx' tracking variable needed anymore!
    """
    # 3 rotations (Rx, Ry, Rz) on each qubit before CNOT
    qml.RX(block_weights[0], wires=q1)
    qml.RY(block_weights[1], wires=q1)
    qml.RZ(block_weights[2], wires=q1)
    
    qml.RX(block_weights[3], wires=q2)
    qml.RY(block_weights[4], wires=q2)
    qml.RZ(block_weights[5], wires=q2)
        
    qml.CNOT(wires=[q1, q2])
    
    # 3 rotations (Rx, Ry, Rz) on each qubit after CNOT
    qml.RX(block_weights[6], wires=q1)
    qml.RY(block_weights[7], wires=q1)
    qml.RZ(block_weights[8], wires=q1)
    
    qml.RX(block_weights[9], wires=q2)
    qml.RY(block_weights[10], wires=q2)
    qml.RZ(block_weights[11], wires=q2)
    
    # Notice: We don't return an idx anymore either!

def apply_sim_ansatz(wires, ansatz_weights, num_layers):
    """
    Applies a flexible number of alternating layers.
    ansatz_weights shape: (total_blocks, 12)
    """
    n = len(wires)
    if n < 2:
        return
        
    block_idx = 0
    for layer in range(num_layers):
        # Even couples: (0, 1), (2, 3), ...
        for i in range(0, n - 1, 2):
            circuit_G_block(wires[i], wires[i+1], ansatz_weights[block_idx])
            block_idx += 1
            
        # Odd couples: (1, 2), (3, 4), ...
        for i in range(1, n - 1, 2):
            circuit_G_block(wires[i], wires[i+1], ansatz_weights[block_idx])
            block_idx += 1

def count_blocks(n_qubits, num_layers):
    """Calculates exactly how many 2-qubit blocks are needed across all layers.
    # Each 2-qubit block uses 12 parameters. Total pairs (odd+even) = (n-1) per layer
    """
    if n_qubits < 2:
        return 0
    even_blocks = len(list(range(0, n_qubits - 1, 2)))
    odd_blocks = len(list(range(1, n_qubits - 1, 2)))
    return num_layers * (even_blocks + odd_blocks)

def create_quantum_transformer_circuit(log_length_sequences, log_feature_dimension, non_linear_order, num_layers):
    # --- (1) Register Setup ---
    wires_C = list(range(log_length_sequences))
    
    start_A = log_length_sequences
    wires_A = list(range(start_A, start_A + log_feature_dimension))
    
    start_B = start_A + log_feature_dimension
    wires_B = []
    for r in range(non_linear_order):
        wires_B.append(list(range(start_B + r * log_feature_dimension, start_B + (r + 1) * log_feature_dimension)))
        
    all_wires_B = [w for reg in wires_B for w in reg]
    all_system_wires = wires_A + all_wires_B
    total_wires = wires_C + all_system_wires
    
    dev = qml.device("default.qubit", wires=total_wires)
    
    blocks_per_ansatz = count_blocks(log_feature_dimension, num_layers)
    params_C_rotations = 3 * log_length_sequences
    
    @qml.qnode(dev, interface="jax", diff_method="backprop")
    def qnode(x, tilde_x, precomputed_states, weights_V, weights_W, weights_C):
        length_sequences = 2**log_length_sequences
        
        # --- (2) OPTIMIZED: Unified Global State Prep ---
        # Instead of 64 loops, stack the vectors along a new axis and flatten.
        # This creates the exact same full 15-qubit superposition in ONE shot.
        global_state = jnp.stack(precomputed_states).flatten() 
        global_state = global_state / jnp.linalg.norm(global_state) # Ensure perfect normalization
        
        qml.StatePrep(global_state, wires=total_wires)
            
        # --- (3) Parameterized Variational Circuits ---
        if blocks_per_ansatz > 0:
            apply_sim_ansatz(wires_A, weights_V, num_layers)
            for r in range(non_linear_order):
                apply_sim_ansatz(wires_B[r], weights_W, num_layers)
                
        # --- (4) OPTIMIZED: Unified Anti-encoding Block ---
        # We can also compress Step 4 by classically computing the combined 
        # target state for each branch, preventing PennyLane loop bloating.
        for j in range(length_sequences):
            binary_j = [int(b) for b in format(j, f'0{log_length_sequences}b')]
            
            # Classically Kronecker the target states for this sequence item
            combined_target = tilde_x[j+1]
            for r in range(non_linear_order):
                combined_target = jnp.kron(combined_target, x[j])
            
            # Only 1 controlled operation per loop step instead of 3 separate ones!
            qml.ctrl(
                qml.adjoint(qml.StatePrep),
                control=wires_C,
                control_values=binary_j
            )(combined_target, wires=all_system_wires)
            
        # --- (5) Measurement Cleanup ---
        idx_C = 0
        for q in wires_C:
            qml.Rot(weights_C[idx_C], weights_C[idx_C+1], weights_C[idx_C+2], wires=q)
            idx_C += 3
            
        for q in wires_C:
            qml.Hadamard(wires=q)
            
        return qml.expval(qml.Projector([0] * len(total_wires), wires=total_wires))

    return qnode, blocks_per_ansatz, params_C_rotations

JAX running on: [CpuDevice(id=0)]


In [ ]:
# =====================================================================
# Example Execution Routine
# =====================================================================
if __name__ == "__main__":
    # Hyperparameters
    log_length_sequences = 5   # Sequence Length T = 2^2 = 4
    log_feature_dimension = 2  # Feature Dim d = 2^2 = 4
    non_linear_order = 2       # Number of B registers
    num_layers = 3             # Number of variational layers
    
    T = 2**log_length_sequences
    d = 2**log_feature_dimension
    
    # Generate dummy classical dataset arrays (normalized amplitude matrices)
    key = jax.random.PRNGKey(42)
    raw_x = jax.random.normal(key, (T, d))
    x = raw_x / jnp.linalg.norm(raw_x, axis=1, keepdims=True)
    
    raw_tilde_x = jax.random.normal(key, (T + 1, d)) # T+1 to safely accommodate j+1 indexing
    tilde_x = raw_tilde_x / jnp.linalg.norm(raw_tilde_x, axis=1, keepdims=True)
    
    # Instantiate circuit builder
    qnode, n_blocks, n_params_C = create_quantum_transformer_circuit(
        log_length_sequences, log_feature_dimension, non_linear_order, num_layers
    )
    
    # Initialize trainable weights
    weights_V = jax.random.uniform(key, (n_blocks, 12), minval=0.0, maxval=2 * jnp.pi)
    weights_W = jax.random.uniform(key, (n_blocks, 12), minval=0.0, maxval=2 * jnp.pi)
    weights_C = jax.random.uniform(key, (n_params_C,), minval=0.0, maxval=2 * jnp.pi)
    
    # Pre-compute static state vectors for state preparations
    precomputed_states = prepare_superposition_vectors(
        x, log_length_sequences, log_feature_dimension, non_linear_order
    )
    
    # Compile the circuit with JAX JIT compilation for lightning-fast local CPU execution
    jitted_qnode = jax.jit(qnode)
    
    # Warmup / First execution (triggers compilation)
    print("Compiling circuit via XLA...")
    start_time = time.time()
    result = jitted_qnode(x, tilde_x, precomputed_states, weights_V, weights_W, weights_C)
    # CRITICAL: Force Python to wait until the GPU/CPU actually finishes the math
    result.block_until_ready() 

    end_time = time.time()
    total_seconds = end_time - start_time

    print(f"Compilation complete.")
    print(f"Initial cross-entropy overlap loss: {result:.6f}")
    print(f"TRUE Wall-Clock Time: {total_seconds:.2f} seconds")
    #print(f"Compilation complete. Initial cross-entropy overlap loss: {result:.6f}. Elapsed time at compilation: {(end_time - start_time)/10 * 1000:.2f} ms")
    
    # Performance check post-compilation

    start_time = time.time()
    for _ in range(10):
        result = jitted_qnode(x, tilde_x, precomputed_states, weights_V, weights_W, weights_C)
    end_time = time.time()
    print(f"Average execution time per step (JIT): {(end_time - start_time)/10 * 1000:.2f} ms")

In [2]:
# ==========================================
# SYNTHETIC TOKEN DATASET GENERATOR
# ==========================================
def generate_synthetic_dataset(batch_size, T, d_dim, non_linear_order, key):
    """Generates physically valid, normalized dense quantum amplitude tokens."""
    k1, k2, k3 = jax.random.split(key, 3)
    
    # Feature inputs: Shape (Batch, Sequence_Length, Dimension)
    raw_x = jax.random.normal(k1, (batch_size, T, d_dim)) + 1j * jax.random.normal(k2, (batch_size, T, d_dim))
    x = raw_x / jnp.linalg.norm(raw_x, axis=-1, keepdims=True)
    
    # Shuffled Target inputs: Shape (Batch, Sequence_Length + 1, Dimension)
    raw_tilde = jax.random.normal(k3, (batch_size, T + 1, d_dim)) + 1j * jax.random.normal(k1, (batch_size, T + 1, d_dim))
    tilde_x = raw_tilde / jnp.linalg.norm(raw_tilde, axis=-1, keepdims=True)
    
    # State Vector initializations: Shape (Batch, Sequence_Length, Global_System_Dimension)
    system_qubits = d_dim ** (non_linear_order + 1)
    raw_states = jax.random.normal(k2, (batch_size, T, system_qubits)) + 1j * jax.random.normal(k3, (batch_size, T, system_qubits))
    precomputed_states = raw_states / jnp.linalg.norm(raw_states, axis=-1, keepdims=True)
    
    return x, tilde_x, precomputed_states

# ==========================================
# LOSS FUNCTION & TRAINING STEPS
# ==========================================
# Hyperparameters
log_length_sequences = 2   # Sequence Length T = 4
log_feature_dimension = 2  # Feature Dim d = 4 qubits (dim=16)
non_linear_order = 1       # 1 Alpha register, 1 Beta register
num_layers = 2             

# Build Base Circuit
qnode, n_blocks, n_params_C = create_quantum_transformer_circuit(
    log_length_sequences, log_feature_dimension, non_linear_order, num_layers
)

# Vectorize the Circuit over the batch axis (0) for all data arguments
# Weights (V, W, C) are marked as None because they are shared across the batch
vmapped_qnode = jax.vmap(qnode, in_axes=(0, 0, 0, None, None, None))

def compute_loss(weights_V, weights_W, weights_C, batch_x, batch_tilde_x, batch_prep):
    """Evaluates Mean Squared Error of the batched quantum projection overlap."""
    overlaps = vmapped_qnode(batch_x, batch_tilde_x, batch_prep, weights_V, weights_W, weights_C)
    return jnp.mean((1.0 - overlaps) ** 2)

@jax.jit
def train_step(weights_V, weights_W, weights_C, batch_x, batch_tilde_x, batch_prep, lr=0.1):
    """Executes a fully JIT-compiled optimization step."""
    loss_val, grads = jax.value_and_grad(compute_loss, argnums=(0, 1, 2))(
        weights_V, weights_W, weights_C, batch_x, batch_tilde_x, batch_prep
    )
    # Basic Gradient Descent Steps
    new_V = weights_V - lr * grads[0]
    new_W = weights_W - lr * grads[1]
    new_C = weights_C - lr * grads[2]
    return loss_val, new_V, new_W, new_C

# ==========================================
# RUNTIME PIPELINE INITIALIZATION
# ==========================================
if __name__ == "__main__":
    master_key = jax.random.PRNGKey(42)
    batch_size = 8
    T_seq = 2**log_length_sequences
    d_dim = 2**log_feature_dimension

    # Run Synthetic Data Generation
    data_key, weight_key = jax.random.split(master_key)
    bx, b_tilde, b_prep = generate_synthetic_dataset(batch_size, T_seq, d_dim, non_linear_order, data_key)

    # Initialize weights
    kv, kw, kc = jax.random.split(weight_key, 3)
    w_V = jax.random.uniform(kv, (n_blocks, 12), minval=0.0, maxval=2 * jnp.pi)
    w_W = jax.random.uniform(kw, (n_blocks, 12), minval=0.0, maxval=2 * jnp.pi)
    w_C = jax.random.uniform(kc, (n_params_C,), minval=0.0, maxval=2 * jnp.pi)

    print("--- Starting Quantum Transformer Training Pipeline ---")
    print(f"Batch Size: {batch_size} | Sequence Length: {T_seq} | Feature Dim: {d_dim}")
    print("Compiling graph execution loops via XLA... (Warmup step)")

    epochs = 5
    for epoch in range(epochs):
        t0 = time.time()
        
        # Execute training iteration step
        loss, w_V, w_W, w_C = train_step(w_V, w_W, w_C, bx, b_tilde, b_prep)
        
        # Enforce strict synchronous block tracking to catch accurate hardware times
        loss.block_until_ready()
        t1 = time.time()
        
        print(f"Epoch {epoch+1:02d}/{epochs:02d} -> Loss value: {loss:.6f} | True execution time: {t1 - t0:.2f}s")

--- Starting Quantum Transformer Training Pipeline ---
Batch Size: 8 | Sequence Length: 4 | Feature Dim: 4
Compiling graph execution loops via XLA... (Warmup step)
Epoch 01/05 -> Loss value: 0.971675 | True execution time: 10.18s
Epoch 02/05 -> Loss value: 0.971409 | True execution time: 0.01s
Epoch 03/05 -> Loss value: 0.971142 | True execution time: 0.00s
Epoch 04/05 -> Loss value: 0.970873 | True execution time: 0.00s
Epoch 05/05 -> Loss value: 0.970603 | True execution time: 0.01s
